In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
!pip install -q ipycanvas ipympl tqdm

import torch
from torch import nn, optim
from torchvision import datasets, transforms
from typing import OrderedDict
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import plotly.graph_objects as go
from collections import OrderedDict
from tqdm.notebook import tqdm
from google.colab import output
output.enable_custom_widget_manager()

from IPython.display import display, clear_output
import ipywidgets as widgets
from ipycanvas import Canvas, hold_canvas
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


## Dataset

In [ ]:
training_set = datasets.MNIST(
    root='./data', train=True, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.view(-1))
    ])
)
test_set = datasets.MNIST(
    root='./data', train=False, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.view(-1))
    ])
)

training_loader = torch.utils.data.DataLoader(training_set, batch_size=4, shuffle=True, num_workers=2)
test_loader     = torch.utils.data.DataLoader(test_set,     batch_size=4, shuffle=False, num_workers=2)


In [ ]:
plt.imshow(test_set[202][0].reshape(28, 28), cmap='gray')
plt.title('Sample MNIST image')
plt.axis('off')
plt.show()


## Autoencoder Architecture

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, layers, activation):
        super(Autoencoder, self).__init__()

        # Encoder
        encoder_layers = []
        for i in range(len(layers) - 1):
            encoder_layers.append(('linear{}'.format(i), nn.Linear(layers[i], layers[i + 1])))
            if i != len(layers) - 2:
                encoder_layers.append(('activation{}'.format(i), activation()))

        # Decoder (mirrors encoder)
        decoder_layers = []
        rev = layers[::-1]
        for i in range(len(rev) - 1):
            decoder_layers.append(('linear{}'.format(i), nn.Linear(rev[i], rev[i + 1])))
            if i < len(layers) - 2:
                decoder_layers.append(('activation{}'.format(i), activation()))
            else:
                decoder_layers.append(('output_activation', nn.Sigmoid()))

        self.encoder = nn.Sequential(OrderedDict(encoder_layers))
        self.decoder = nn.Sequential(OrderedDict(decoder_layers))

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        return self.decoder(self.encoder(x))


# Example architecture
Autoencoder([28*28, 256, 128, 64, 32, 16, 10], nn.ReLU)


## Training Utilities

In [ ]:
def training_run(layers, activation, EPOCHS):
    model = Autoencoder(layers, activation).to(device)
    loss_fn  = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-8)

    total_batches = EPOCHS * len(training_loader)
    with tqdm(total=total_batches, desc="Training") as pbar:
        for epoch in range(EPOCHS):
            model.train(True)
            running_loss = 0.
            for i, (inputs, _) in enumerate(training_loader):
                inputs = inputs.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = loss_fn(outputs, inputs)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
                pbar.update(1)
                pbar.set_postfix(epoch=epoch + 1, loss=f"{running_loss / (i + 1):.5f}")
            model.train(False)

    return model


In [ ]:
def visualize_inference(model, n_example):
    test_image, _ = test_set[n_example]
    test_image = test_image.to(device)

    model.eval()
    with torch.no_grad():
        reconstructed = model(test_image)

    reconstructed = reconstructed.cpu().view(28, 28)
    test_image    = test_image.cpu().view(28, 28)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(test_image.numpy(), cmap='gray')
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    axes[1].imshow(reconstructed.numpy(), cmap='gray')
    axes[1].set_title('Reconstructed Image')
    axes[1].axis('off')
    plt.show()


## First Autoencoder — 10-Neuron Coding Layer

In [ ]:
EPOCHS = 1
model = training_run([28*28, 256, 128, 64, 10], nn.ReLU, EPOCHS)


In [ ]:
for i in range(10):
    visualize_inference(model, i)


### Reconstruction from Corrupted Input
Choose a test image with the slider, then **paint black** with the mouse to erase parts of the digit and see how the autoencoder reconstructs it.

In [ ]:
# ── Interactive canvas: pick a dataset image and paint it black ──────────────
# Use the slider to choose any test-set image, then draw with the mouse to
# erase parts of the digit and observe how the autoencoder reconstructs it.

CANVAS_SIZE = 280
CELL_SIZE   = CANVAS_SIZE // 28   # 10 px per MNIST pixel

# Python-side image state (avoids async sync issues with get_image_data)
_current_img = np.zeros((28, 28), dtype=np.float32)

canvas_draw = Canvas(width=CANVAS_SIZE, height=CANVAS_SIZE)

fig_c, (ax_orig_c, ax_recon_c) = plt.subplots(1, 2, figsize=(8, 4))
img_plot_c   = ax_orig_c.imshow(np.zeros((28, 28)), cmap='gray', vmin=0, vmax=1)
recon_plot_c = ax_recon_c.imshow(np.zeros((28, 28)), cmap='gray', vmin=0, vmax=1)
ax_orig_c.set_title('Modified Image');     ax_orig_c.axis('off')
ax_recon_c.set_title('Reconstruction');   ax_recon_c.axis('off')
plt.tight_layout()
plt.close(fig_c)  # Prevent auto-display; we show it inside fig_out_c

fig_out_c = widgets.Output()


def _render_img_to_canvas():
    """Render _current_img (28x28 float) onto the 280x280 canvas."""
    rgba = np.zeros((CANVAS_SIZE, CANVAS_SIZE, 4), dtype=np.uint8)
    for r in range(28):
        for c in range(28):
            v = int(_current_img[r, c] * 255)
            rgba[r*CELL_SIZE:(r+1)*CELL_SIZE, c*CELL_SIZE:(c+1)*CELL_SIZE] = [v, v, v, 255]
    canvas_draw.put_image_data(rgba, 0, 0)


def load_image_to_canvas(idx):
    """Load a 28x28 MNIST image into _current_img and render it."""
    global _current_img
    img_tensor, _ = test_set[idx]
    _current_img = img_tensor.reshape(28, 28).numpy().copy()
    _render_img_to_canvas()


def update_reconstruction_c():
    """Reconstruct from _current_img (no canvas sync needed)."""
    t = torch.tensor(_current_img, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        recon = model(t.view(-1, 28 * 28)).cpu().view(28, 28).numpy()
    img_plot_c.set_data(_current_img)
    recon_plot_c.set_data(recon)
    fig_c.canvas.draw_idle()
    with fig_out_c:
        clear_output(wait=True)
        display(fig_c)


def _paint_black(x, y, radius=14, render=True):
    """Zero out pixels in _current_img within brush radius and update canvas."""
    col_center = x / CELL_SIZE
    row_center = y / CELL_SIZE
    r_cells = radius / CELL_SIZE
    r_cells_sq = r_cells * r_cells
    changed = False
    for r in range(28):
        for c in range(28):
            if (r + 0.5 - row_center)**2 + (c + 0.5 - col_center)**2 <= r_cells_sq:
                if _current_img[r, c] > 0:
                    _current_img[r, c] = 0.0
                    changed = True
    if changed:
        if render:
            _render_img_to_canvas()
        else:
            _needs_render_c[0] = True
    return changed


_drawing_c = False
_last_t_c  = [0.0]
_needs_render_c = [False]


def on_down_c(x, y):
    global _drawing_c
    _drawing_c = True
    _paint_black(x, y, render=True)
    update_reconstruction_c()


def on_up_c(x, y):
    global _drawing_c
    _drawing_c = False
    # Flush any pending canvas render on release
    if _needs_render_c[0]:
        _needs_render_c[0] = False
        _render_img_to_canvas()
    update_reconstruction_c()


def on_move_c(x, y):
    if _drawing_c:
        now = time.time()
        if now - _last_t_c[0] > 0.10:
            _last_t_c[0] = now
            _paint_black(x, y, render=True)
            update_reconstruction_c()
        else:
            _paint_black(x, y, render=False)


def on_out_c(x, y):
    """Reset drawing state when mouse leaves the canvas."""
    global _drawing_c
    if _drawing_c:
        _drawing_c = False
        if _needs_render_c[0]:
            _needs_render_c[0] = False
            _render_img_to_canvas()
        update_reconstruction_c()


canvas_draw.on_mouse_down(on_down_c)
canvas_draw.on_mouse_up(on_up_c)
canvas_draw.on_mouse_move(on_move_c)
canvas_draw.on_mouse_out(on_out_c)

idx_slider_c = widgets.IntSlider(
    value=0, min=0, max=len(test_set) - 1,
    description='Image index:', layout={'width': '420px'},
    style={'description_width': 'initial'}
)
reset_btn_c = widgets.Button(description='Reset Image', button_style='warning', icon='refresh')


def on_idx_change_c(change):
    load_image_to_canvas(change['new'])
    update_reconstruction_c()


def on_reset_c(b):
    load_image_to_canvas(idx_slider_c.value)
    update_reconstruction_c()


idx_slider_c.observe(on_idx_change_c, names='value')
reset_btn_c.on_click(on_reset_c)

load_image_to_canvas(0)
update_reconstruction_c()

display(widgets.VBox([
    idx_slider_c,
    reset_btn_c,
    widgets.HBox([canvas_draw, fig_out_c])
]))

### Draw Your Own Digit
**Draw** on the blank canvas with the mouse and see how the autoencoder reconstructs your drawing in real time.  
Use the brush-size slider to adjust the pen width, and click **Clear** to start over.

In [ ]:
# ── Interactive canvas: draw your own digit and see the reconstruction ────

DRAW_CANVAS_SIZE = 280
DRAW_CELL_SIZE   = DRAW_CANVAS_SIZE // 28   # 10 px per MNIST pixel

# Python-side image state (28x28, black = 0)
_draw_img = np.zeros((28, 28), dtype=np.float32)

canvas_free = Canvas(width=DRAW_CANVAS_SIZE, height=DRAW_CANVAS_SIZE)
canvas_free.fill_style = 'black'
canvas_free.fill_rect(0, 0, DRAW_CANVAS_SIZE, DRAW_CANVAS_SIZE)

fig_d, (ax_draw_d, ax_recon_d) = plt.subplots(1, 2, figsize=(8, 4))
img_plot_d   = ax_draw_d.imshow(np.zeros((28, 28)), cmap='gray', vmin=0, vmax=1)
recon_plot_d = ax_recon_d.imshow(np.zeros((28, 28)), cmap='gray', vmin=0, vmax=1)
ax_draw_d.set_title('Your Drawing');  ax_draw_d.axis('off')
ax_recon_d.set_title('Reconstruction'); ax_recon_d.axis('off')
plt.tight_layout()
plt.close(fig_d)

fig_out_d = widgets.Output()


def _render_draw_canvas():
    """Render _draw_img (28x28 float) onto the 280x280 canvas."""
    rgba = np.zeros((DRAW_CANVAS_SIZE, DRAW_CANVAS_SIZE, 4), dtype=np.uint8)
    for r in range(28):
        for c in range(28):
            v = int(_draw_img[r, c] * 255)
            rgba[r*DRAW_CELL_SIZE:(r+1)*DRAW_CELL_SIZE,
                 c*DRAW_CELL_SIZE:(c+1)*DRAW_CELL_SIZE] = [v, v, v, 255]
    canvas_free.put_image_data(rgba, 0, 0)


def _update_recon_d():
    """Run the autoencoder on _draw_img and update the plots."""
    t = torch.tensor(_draw_img, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        recon = model(t.view(-1, 28 * 28)).cpu().view(28, 28).numpy()
    img_plot_d.set_data(_draw_img)
    recon_plot_d.set_data(recon)
    fig_d.canvas.draw_idle()
    with fig_out_d:
        clear_output(wait=True)
        display(fig_d)


def _paint_white(x, y, radius, render=True):
    """Set pixels in _draw_img to white within brush radius."""
    col_center = x / DRAW_CELL_SIZE
    row_center = y / DRAW_CELL_SIZE
    r_cells = radius / DRAW_CELL_SIZE
    r_cells_sq = r_cells * r_cells
    changed = False
    for r in range(28):
        for c in range(28):
            if (r + 0.5 - row_center)**2 + (c + 0.5 - col_center)**2 <= r_cells_sq:
                if _draw_img[r, c] < 1.0:
                    _draw_img[r, c] = 1.0
                    changed = True
    if changed:
        if render:
            _render_draw_canvas()
        else:
            _needs_render_d[0] = True
    return changed


_drawing_d = False
_last_t_d  = [0.0]
_needs_render_d = [False]

brush_slider = widgets.IntSlider(
    value=14, min=5, max=40, step=1,
    description='Brush size:', layout={'width': '300px'},
    style={'description_width': 'initial'}
)


def _on_down_d(x, y):
    global _drawing_d
    _drawing_d = True
    _paint_white(x, y, radius=brush_slider.value, render=True)
    _update_recon_d()


def _on_up_d(x, y):
    global _drawing_d
    _drawing_d = False
    if _needs_render_d[0]:
        _needs_render_d[0] = False
        _render_draw_canvas()
    _update_recon_d()


def _on_move_d(x, y):
    if _drawing_d:
        now = time.time()
        if now - _last_t_d[0] > 0.10:
            _last_t_d[0] = now
            _paint_white(x, y, radius=brush_slider.value, render=True)
            _update_recon_d()
        else:
            _paint_white(x, y, radius=brush_slider.value, render=False)


def _on_out_d(x, y):
    global _drawing_d
    if _drawing_d:
        _drawing_d = False
        if _needs_render_d[0]:
            _needs_render_d[0] = False
            _render_draw_canvas()
        _update_recon_d()


canvas_free.on_mouse_down(_on_down_d)
canvas_free.on_mouse_up(_on_up_d)
canvas_free.on_mouse_move(_on_move_d)
canvas_free.on_mouse_out(_on_out_d)

clear_btn_d = widgets.Button(description='Clear', button_style='danger', icon='trash')


def _on_clear_d(b):
    global _draw_img
    _draw_img = np.zeros((28, 28), dtype=np.float32)
    _render_draw_canvas()
    _update_recon_d()


clear_btn_d.on_click(_on_clear_d)

# Initial render
_render_draw_canvas()
_update_recon_d()

display(widgets.VBox([
    widgets.HBox([brush_slider, clear_btn_d]),
    widgets.HBox([canvas_free, fig_out_d])
]))

### Exploring the Coding Layer Neurons

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, VBox, HBox, Button, Output
from IPython.display import display, clear_output

n = 10

# ── Pre-compute: extract decoder weights/biases to numpy ──────────────────
# Avoids torch tensor creation and forward-pass overhead on every slider move.
model.eval()
_dec_layers = []
for layer in model.decoder:
    if isinstance(layer, torch.nn.Linear):
        _dec_layers.append(('linear',
                            layer.weight.data.cpu().numpy(),
                            layer.bias.data.cpu().numpy()))
    elif isinstance(layer, torch.nn.ReLU):
        _dec_layers.append(('relu', None, None))
    elif isinstance(layer, torch.nn.Sigmoid):
        _dec_layers.append(('sigmoid', None, None))

def _fast_decode(vec):
    """Decoder forward pass using pre-extracted numpy weights."""
    x = vec
    for kind, w, b in _dec_layers:
        if kind == 'linear':
            x = x @ w.T + b
        elif kind == 'relu':
            x = np.maximum(x, 0)
        elif kind == 'sigmoid':
            x = 1.0 / (1.0 + np.exp(-x))
    return x.reshape(28, 28)

# ── Pre-create figure (reused on every slider update) ─────────────────────
# Avoids creating a new plt.figure() on each change.
_fig_sl, _ax_sl = plt.subplots(figsize=(6, 6))
_im_sl = _ax_sl.imshow(np.zeros((28, 28)), cmap='gray', vmin=0, vmax=1)
_ax_sl.set_title('Generated Image')
_ax_sl.axis('off')
plt.close(_fig_sl)

_out_sl = Output()

def update_image(**kwargs):
    vec = np.array([kwargs[f'component_{i}'] for i in range(n)], dtype=np.float32)
    _im_sl.set_data(_fast_decode(vec))
    with _out_sl:
        clear_output(wait=True)
        display(_fig_sl)
        print(f"Current vector: {vec}")

sliders = {}
for i in range(n):
    initial_value = 100.0 if i in [3, 9] else 0.0
    sliders[f'component_{i}'] = FloatSlider(
        value=initial_value, min=-10.0, max=10.0, step=0.01,
        description=f'Neurona {i + 1}:',
        style={'description_width': 'initial'},
        layout={'width': '400px'}
    )

reset_button = Button(description='Resetear a Cero', button_style='warning', icon='refresh')

def on_reset_clicked(b):
    for i in range(n):
        sliders[f'component_{i}'].value = 0.0

reset_button.on_click(on_reset_clicked)

def _on_slider_change(change):
    update_image(**{k: s.value for k, s in sliders.items()})

for slider in sliders.values():
    slider.observe(_on_slider_change, names='value')

# Initial render
update_image(**{k: s.value for k, s in sliders.items()})

left_column   = VBox([sliders[f'component_{i}'] for i in range(5)])
right_column  = VBox([sliders[f'component_{i}'] for i in range(5, 10)])
slider_layout = HBox([left_column, right_column])
controls      = VBox([slider_layout, reset_button])
ui            = HBox([controls, _out_sl])
display(ui)

In [ ]:
import torch
import torch.nn.functional as F

n_code   = 10
n_digits = 10

all_activations = [[] for _ in range(n_digits)]

print("Collecting activations from the test set...")
model.eval()
with torch.no_grad():
    for data, labels in test_loader:
        data = data.to(device)
        code_vectors     = model.encoder(data)
        code_vectors_cpu = code_vectors.cpu()
        labels_cpu       = labels.cpu()
        for i in range(len(labels_cpu)):
            all_activations[labels_cpu[i].item()].append(code_vectors_cpu[i].abs())

print("Collection complete.")

avg_activations_list = []
for digit in range(n_digits):
    if all_activations[digit]:
        avg_activations_list.append(torch.stack(all_activations[digit]).mean(dim=0))
    else:
        avg_activations_list.append(torch.zeros(n_code))

avg_activations_matrix = torch.stack(avg_activations_list)

importance_matrix = torch.zeros_like(avg_activations_matrix)
for i in range(n_digits):
    row = avg_activations_matrix[i]
    importance_matrix[i] = (row - row.min()) / (row.max() - row.min() + 1e-6)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

sns.heatmap(
    avg_activations_matrix.numpy(), annot=True, fmt=".2f", cmap='viridis',
    xticklabels=[f"Neuron {i+1}" for i in range(n_code)],
    yticklabels=[f"Digit {i}" for i in range(n_digits)],
    ax=axes[0]
)
axes[0].set_title("Raw Average Activations by Digit")
axes[0].set_xlabel("Code Layer Neurons")
axes[0].set_ylabel("Input Digit Class")

sns.heatmap(
    importance_matrix.numpy(), annot=True, fmt=".2f", cmap='viridis',
    xticklabels=[f"Neuron {i+1}" for i in range(n_code)],
    yticklabels=[f"Digit {i}" for i in range(n_digits)],
    ax=axes[1]
)
axes[1].set_title("Normalized Neuron Importance by Digit")
axes[1].set_xlabel("Code Layer Neurons")
axes[1].set_ylabel("Input Digit Class")

plt.tight_layout()
plt.show()


In [ ]:
first_layer_weights = model.encoder[0].weight.data.cpu()
print(f"First layer weights shape: {first_layer_weights.shape}")

num_to_show = 256
grid_size   = int(num_to_show ** 0.5)   # 16

fig, axes = plt.subplots(grid_size, grid_size, figsize=(16, 16))
fig.suptitle('First Layer Weights (All 256 Neurons)', fontsize=20)

for i, ax in enumerate(axes.flat):
    if i < num_to_show:
        weight_image = first_layer_weights[i].reshape(28, 28)
        ax.imshow(weight_image, cmap='gray')
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


## Second Autoencoder — 3-Neuron Coding Layer

In [ ]:
EPOCHS = 1
model_2 = training_run([28*28, 256, 128, 64, 3], nn.ReLU, EPOCHS)


In [ ]:
for i in range(10):
    visualize_inference(model_2, i)


In [ ]:
import torch
import torch.nn.functional as F

n_code_2  = 3
n_digits  = 10
target_model = model_2

all_activations_2 = [[] for _ in range(n_digits)]

print(f"Recolectando activaciones del modelo con {n_code_2} neuronas...")
target_model.eval()
with torch.no_grad():
    for data, labels in test_loader:
        data = data.to(device)
        code_vectors     = target_model.encoder(data)
        code_vectors_cpu = code_vectors.cpu()
        labels_cpu       = labels.cpu()
        for i in range(len(labels_cpu)):
            all_activations_2[labels_cpu[i].item()].append(code_vectors_cpu[i].abs())

print("Recolección completa.")

avg_activations_list_2 = []
for digit in range(n_digits):
    if all_activations_2[digit]:
        avg_activations_list_2.append(torch.stack(all_activations_2[digit]).mean(dim=0))
    else:
        avg_activations_list_2.append(torch.zeros(n_code_2))

avg_activations_matrix_2 = torch.stack(avg_activations_list_2)

importance_matrix_3d = torch.zeros_like(avg_activations_matrix_2)
for i in range(n_digits):
    row = avg_activations_matrix_2[i]
    importance_matrix_3d[i] = (row - row.min()) / (row.max() - row.min() + 1e-6)

print("Matriz de importancia calculada.")


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interactive_output, FloatSlider, VBox, HBox, Button, Output
from IPython.display import display

n_code_2 = 3
n_digits  = 10

sliders_3d = {}
for i in range(n_code_2):
    sliders_3d[f'component_3d_{i}'] = FloatSlider(
        value=0.0, min=-4.0, max=4.0, step=0.01,
        description=f'Neurona {i + 1}:',
        style={'description_width': 'initial'},
        layout={'width': '300px'}
    )

reset_button_3d = Button(description='Resetear a Cero', button_style='warning', icon='refresh')

def on_reset_3d_clicked(b):
    for i in range(n_code_2):
        sliders_3d[f'component_3d_{i}'].value = 0.0

reset_button_3d.on_click(on_reset_3d_clicked)

def update_image_3d(**kwargs):
    input_vector = np.array([kwargs[f'component_3d_{i}'] for i in range(n_code_2)])
    input_tensor = torch.from_numpy(input_vector).float().to(device)
    out = model_2.decoder(input_tensor)
    test_image = out.cpu().view(28, 28)
    plt.figure(figsize=(6, 6))
    plt.imshow(test_image.detach().numpy(), cmap='gray')
    plt.title('Imagen Generada (3 Neuronas)')
    plt.axis('off')
    plt.show()
    print(f"Vector actual (3D): {input_vector}")

importance_heatmap_output = Output()
with importance_heatmap_output:
    plt.figure(figsize=(6, 8))
    sns.heatmap(
        importance_matrix_3d.numpy(), annot=True, fmt=".2f", cmap='viridis',
        xticklabels=[f"Neurona {i+1}" for i in range(n_code_2)],
        yticklabels=[f"Dígito {i}" for i in range(n_digits)]
    )
    plt.title(f"Importancia (Capa = {n_code_2})")
    plt.xlabel("Neuronas")
    plt.ylabel("Dígito")
    plt.show()

avg_heatmap_output = Output()
with avg_heatmap_output:
    plt.figure(figsize=(6, 8))
    sns.heatmap(
        avg_activations_matrix_2.numpy(), annot=True, fmt=".2f", cmap='viridis',
        xticklabels=[f"Neurona {i+1}" for i in range(n_code_2)],
        yticklabels=[f"Dígito {i}" for i in range(n_digits)]
    )
    plt.title("Activación Promedio por Dígito")
    plt.xlabel("Neuronas")
    plt.ylabel("Dígito")
    plt.show()

plot_output_3d    = interactive_output(update_image_3d, sliders_3d)
slider_col        = VBox(list(sliders_3d.values()) + [reset_button_3d])
controls_3d       = HBox([slider_col, plot_output_3d])
ui_3d             = VBox([controls_3d, HBox([importance_heatmap_output, avg_heatmap_output])])
display(ui_3d)


In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go
from IPython.display import display

# --- 1. Configurar los "Hooks" (Espías) en model_2 ---

activations = {}

def get_activation(name):
    """Crea una función hook para capturar activaciones."""
    def hook(model, input, output):
        activations[name] = output.detach().cpu()
    return hook

# Asegúrate de que los hooks están en el modelo correcto: model_2
# Y asume que el codificador es un iterable (como nn.Sequential)
for i, encoder_layer in enumerate(model_2.encoder):
    encoder_layer.register_forward_hook(get_activation(f'encoder layer {i}'))

# ¡IMPORTANTE! Necesitamos el índice de la capa de código.
# Asumimos que es la ÚLTIMA capa del codificador.
# (Si tu encoder tiene 4 capas, sus índices son 0, 1, 2, 3. len=4, así que el índice es 3)
n_code_layer_index = len(model_2.encoder) - 1
print(f"Espiando la capa del codificador: {n_code_layer_index}")


# --- 2. Recolectar Activaciones de model_2 ---

# Inicializa 10 listas, una para cada dígito
X = [[] for _ in range(10)]
Y = [[] for _ in range(10)] # Para mantener un seguimiento, aunque no se use en el gráfico

print("Recolectando activaciones del dataset...")
model_2.eval() # Poner model_2 en modo de evaluación

# Procesar las primeras 5000 muestras
for j in range(10000):
    test_image, label = test_set[j]
    # Mueve la imagen al device y añade una dimensión de 'batch' (lote)
    test_image = test_image.to(device).unsqueeze(0)

    with torch.no_grad():
        # Ejecuta la inferencia en model_2
        # Esto disparará los hooks que acabamos de registrar
        reconstructed_image = model_2(test_image)

    # Obtiene la activación de la capa de código que nos interesa
    act = activations[f"encoder layer {n_code_layer_index}"]

    # .squeeze() elimina la dimensión extra del 'batch'
    features = act.squeeze().tolist()

    # Añade el vector de características (3D) a la lista del dígito correcto
    X[label].append(features)
    Y[label].append(label)

print("Recolección completa.")

# --- 3. Crear el Diccionario de Datos para Graficar ---

digit_datasets = {}
for digit in range(10):
    digit_datasets[digit] = {
        'activation_vectors': np.array(X[digit]),
        'labels': np.array(Y[digit])
    }

# Imprime un resumen de los datos recolectados
for digit, data in digit_datasets.items():
    print(f"Dígito {digit}: {len(data['activation_vectors'])} muestras, "
          f"Forma (Shape): {data['activation_vectors'].shape}")


# --- 4. Graficar con Plotly (Tu código, sin cambios) ---

# Define una lista de colores para cada dígito
colors = ['red', 'green', 'blue', 'cyan', 'magenta',
          'yellow', 'black', 'orange', 'purple', 'brown']

fig = go.Figure()

for digit, data in digit_datasets.items():
    activation_vectors = data['activation_vectors']

    # Asegúrate de que hay datos antes de intentar graficar
    if len(activation_vectors) > 0:
        fig.add_trace(go.Scatter3d(
            x=activation_vectors[:, 0], # Componente 1 (X)
            y=activation_vectors[:, 1], # Componente 2 (Y)
            z=activation_vectors[:, 2], # Componente 3 (Z)
            mode='markers',
            marker=dict(size=3, color=colors[digit]),
            name=f'Dígito {digit}'
        ))

fig.update_layout(
    title="Espacio de Activación 3D por Dígito (Modelo de 3 Neuronas)",
    scene=dict(
        xaxis_title="Neurona 1",
        yaxis_title="Neurona 2",
        zaxis_title="Neurona 3"
    ),
    width=900,
    height=700
)

fig.show()

In [ ]:
# --- 5. Calcular y Graficar Vectores Promedio ---

# 5.a. Calcular el vector de activación promedio para cada dígito
average_vectors = {}
for digit, data in digit_datasets.items():
    if len(data['activation_vectors']) > 0:
        # Calcula la media de todas las activaciones para este dígito
        avg_vec = np.mean(data['activation_vectors'], axis=0)
        average_vectors[digit] = avg_vec
    else:
        # En caso de que no se hayan encontrado muestras (poco probable)
        average_vectors[digit] = np.array([0, 0, 0])

print("\nVectores de activación promedio:")
for digit, avg_vec in average_vectors.items():
    print(f"Dígito {digit}: {avg_vec}")

# 5.b. Crear la segunda gráfica 3D para los vectores promedio
fig_avg = go.Figure()

# Reutilizamos la misma lista de colores
colors = ['red', 'green', 'blue', 'cyan', 'magenta',
          'yellow', 'black', 'orange', 'purple', 'brown']

for digit, avg_vec in average_vectors.items():
    # Para dibujar una flecha desde (0,0,0) hasta (x,y,z),
    # simplemente trazamos una línea entre esos dos puntos.
    fig_avg.add_trace(go.Scatter3d(
        x=[0, avg_vec[0]],  # Coordenadas X: [inicio, fin]
        y=[0, avg_vec[1]],  # Coordenadas Y: [inicio, fin]
        z=[0, avg_vec[2]],  # Coordenadas Z: [inicio, fin]
        mode='lines+markers',
        line=dict(color=colors[digit], width=6), # Línea más gruesa
        marker=dict(size=4, color=colors[digit]), # Marcador en la punta
        name=f'Promedio Dígito {digit}'
    ))

# 5.c. Configurar el layout de la nueva gráfica
fig_avg.update_layout(
    title="Vectores de Activación Promedio por Dígito (Desde el Origen)",
    scene=dict(
        xaxis_title="Neurona 1",
        yaxis_title="Neurona 2",
        zaxis_title="Neurona 3",
        # Opcional: Asegurar que los ejes tengan un rango similar
        # aspectmode='cube'
    ),
    width=900,
    height=700
)

# 5.d. Mostrar la nueva gráfica
fig_avg.show()

### Visualización de Transformaciones Capa por Capa
Proyección 2D de las activaciones en cada capa del encoder usando PCA y UMAP. Esto permite ver cómo el autoencoder va separando los dígitos capa por capa.

In [ ]:
# --- Recolectar activaciones de TODAS las capas del encoder de model_2 ---

_layer_acts = {}

def _make_hook(name):
    def hook(module, inp, out):
        if name not in _layer_acts:
            _layer_acts[name] = []
        _layer_acts[name].append(out.detach().cpu())
    return hook

_hooks = []
for i, layer in enumerate(model_2.encoder):
    _hooks.append(layer.register_forward_hook(_make_hook(i)))

_all_labels = []
_all_inputs = []

print("Recolectando activaciones de todas las capas del encoder...")
model_2.eval()
with torch.no_grad():
    for data, labels in test_loader:
        data = data.to(device)
        _ = model_2(data)
        _all_inputs.append(data.cpu())
        _all_labels.append(labels)

# Remover hooks
for h in _hooks:
    h.remove()

# Concatenar
all_labels_arr = torch.cat(_all_labels).numpy()
all_inputs_arr = torch.cat(_all_inputs).numpy()

# Construir lista ordenada: (nombre, array de activaciones)
layer_names_vis = ["Input (784D)"]
layer_data_vis = [all_inputs_arr]

for i, mod_name in enumerate(model_2.encoder._modules.keys()):
    acts = torch.cat(_layer_acts[i]).numpy()
    mod = model_2.encoder[i]
    if isinstance(mod, nn.Linear):
        label = f"{mod_name}: Linear {mod.in_features}→{mod.out_features}"
    else:
        label = f"{mod_name}: ReLU ({acts.shape[1]}D)"
    layer_names_vis.append(label)
    layer_data_vis.append(acts)

print(f"\nCapas recolectadas: {len(layer_names_vis)}")
for name, d in zip(layer_names_vis, layer_data_vis):
    print(f"  {name}: shape {d.shape}")

In [ ]:
from sklearn.decomposition import PCA

# --- PCA 2D: Transformación capa por capa ---

n_stages = len(layer_names_vis)
n_cols = 4
n_rows = (n_stages + n_cols - 1) // n_cols

colors_vis = ['red', 'green', 'blue', 'cyan', 'magenta',
              'yellow', 'black', 'orange', 'purple', 'brown']

fig_pca, axes_pca = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes_flat_p = axes_pca.flatten()

for idx in range(n_stages):
    ax = axes_flat_p[idx]
    data = layer_data_vis[idx]

    if data.shape[1] >= 2:
        proj = PCA(n_components=2).fit_transform(data)
    else:
        proj = np.column_stack([data, np.zeros(len(data))])

    for digit in range(10):
        mask = all_labels_arr == digit
        ax.scatter(proj[mask, 0], proj[mask, 1],
                   c=colors_vis[digit], s=1, alpha=0.4, label=str(digit))

    ax.set_title(layer_names_vis[idx], fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

for idx in range(n_stages, len(axes_flat_p)):
    axes_flat_p[idx].set_visible(False)

handles, labels_leg = axes_flat_p[0].get_legend_handles_labels()
fig_pca.legend(handles, labels_leg, loc='lower right', ncol=5, markerscale=8, fontsize=11)
fig_pca.suptitle("PCA 2D — Transformación capa por capa (model_2: 784→256→128→64→3)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
!pip install -q umap-learn
import umap

# --- UMAP 2D: Transformación capa por capa ---

n_stages = len(layer_names_vis)
n_cols = 4
n_rows = (n_stages + n_cols - 1) // n_cols

colors_vis = ['red', 'green', 'blue', 'cyan', 'magenta',
              'yellow', 'black', 'orange', 'purple', 'brown']

fig_umap, axes_umap = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes_flat_u = axes_umap.flatten()

for idx in range(n_stages):
    ax = axes_flat_u[idx]
    data = layer_data_vis[idx]
    print(f"UMAP en capa {idx + 1}/{n_stages}: {layer_names_vis[idx]}...")

    if data.shape[1] >= 2:
        proj = umap.UMAP(n_components=2, random_state=42).fit_transform(data)
    else:
        proj = np.column_stack([data, np.zeros(len(data))])

    for digit in range(10):
        mask = all_labels_arr == digit
        ax.scatter(proj[mask, 0], proj[mask, 1],
                   c=colors_vis[digit], s=1, alpha=0.4, label=str(digit))

    ax.set_title(layer_names_vis[idx], fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

for idx in range(n_stages, len(axes_flat_u)):
    axes_flat_u[idx].set_visible(False)

handles, labels_leg = axes_flat_u[0].get_legend_handles_labels()
fig_umap.legend(handles, labels_leg, loc='lower right', ncol=5, markerscale=8, fontsize=11)
fig_umap.suptitle("UMAP 2D — Transformación capa por capa (model_2: 784→256→128→64→3)", fontsize=14)
plt.tight_layout()
plt.show()

## Further Experiments
Train a model with 16 coding neurons and inspect its reconstructions.

In [ ]:
EPOCHS = 1
model_3 = training_run([28*28, 256, 128, 64, 32, 16], nn.ReLU, EPOCHS)

for i in range(10):
    visualize_inference(model_3, i)


## Digit-Specific Autoencoders
Train a separate autoencoder for each digit (0-9). Each model is trained exclusively on images of its target digit, learning digit-specific reconstruction features. Then visualize the latent space with interactive 2D PCA, 2D UMAP, and hover-to-see reconstructions.

In [ ]:
# ==================================================================
# TRAIN DIGIT-SPECIFIC AUTOENCODERS (one per digit, 0-9)
# ==================================================================

digit_autoencoders = {}
EPOCHS_DIGIT = 1
DIGIT_LAYERS = [28*28, 256, 128, 64, 10]

for digit in range(10):
    print(f"\n{'='*55}")
    print(f"  Training autoencoder for digit {digit}")
    print(f"{'='*55}")

    # Filter training set for this digit only
    digit_indices = (training_set.targets == digit).nonzero(as_tuple=True)[0].tolist()
    digit_subset = torch.utils.data.Subset(training_set, digit_indices)
    digit_loader = torch.utils.data.DataLoader(
        digit_subset, batch_size=4, shuffle=True, num_workers=2
    )

    # Create and train
    model_d = Autoencoder(DIGIT_LAYERS, nn.ReLU).to(device)
    loss_fn_d = nn.MSELoss()
    optimizer_d = optim.Adam(model_d.parameters(), lr=1e-3, weight_decay=1e-8)

    total_batches = EPOCHS_DIGIT * len(digit_loader)
    with tqdm(total=total_batches, desc=f"Digit {digit}") as pbar:
        for epoch in range(EPOCHS_DIGIT):
            model_d.train()
            running_loss = 0.
            for i, (inputs, _) in enumerate(digit_loader):
                inputs = inputs.to(device)
                optimizer_d.zero_grad()
                outputs = model_d(inputs)
                loss = loss_fn_d(outputs, inputs)
                loss.backward()
                optimizer_d.step()
                running_loss += loss.item()
                pbar.update(1)
                pbar.set_postfix(loss=f"{running_loss / (i + 1):.5f}")
            model_d.eval()

    digit_autoencoders[digit] = model_d
    print(f"  Samples: {len(digit_indices)}, Final loss: {running_loss / (i + 1):.5f}")

print(f"\n{'='*55}")
print("  All 10 digit-specific autoencoders trained!")
print(f"{'='*55}")

In [ ]:
# ==================================================================
# INTERACTIVE VISUALIZATION — Digit-Specific Autoencoders
# Select a digit to see 2D PCA, 2D UMAP, and hover reconstructions
# ==================================================================
import numpy as np
import umap
from sklearn.decomposition import PCA
import plotly.graph_objects as go
from IPython.display import HTML, display, clear_output
import base64, io
from PIL import Image
import ipywidgets as widgets

_ds_colors = {
    0: '#1f77b4', 1: '#ff7f0e', 2: '#2ca02c', 3: '#d62728', 4: '#9467bd',
    5: '#8c564b', 6: '#e377c2', 7: '#7f7f7f', 8: '#bcbd22', 9: '#17becf'
}

def _flat_to_b64(flat_img):
    """Convert a flat (784,) float image to a base64 PNG string."""
    arr = (flat_img.reshape(28, 28) * 255).astype(np.uint8)
    pil = Image.fromarray(arr).resize((112, 112), Image.LANCZOS)
    buf = io.BytesIO()
    pil.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode()


def _build_digit_viz(digit):
    """Build complete HTML visualization for one digit's autoencoder."""
    ae = digit_autoencoders[digit]
    ae.eval()
    color = _ds_colors[digit]

    # Collect test samples for this digit
    digit_idx = (test_set.targets == digit).nonzero(as_tuple=True)[0].tolist()
    n_samples = min(1500, len(digit_idx))
    chosen = np.random.choice(digit_idx, n_samples, replace=False)
    imgs = np.array([test_set[idx][0].numpy() for idx in chosen])

    # Forward pass through digit-specific autoencoder
    t = torch.FloatTensor(imgs).to(device)
    with torch.no_grad():
        enc = ae.encoder(t)
        dec = ae.decoder(enc)
    acts = enc.cpu().numpy()
    recs = dec.cpu().numpy()

    # PCA 2D
    pca2 = PCA(n_components=2).fit(acts)
    p2 = pca2.transform(acts)

    # UMAP 2D
    print(f"  Computing UMAP 2D embedding...")
    reducer = umap.UMAP(n_components=2, random_state=42)
    u2 = reducer.fit_transform(acts)

    # Convert images to base64
    print(f"  Converting {n_samples} images to base64...")
    orig_b64 = [_flat_to_b64(im) for im in imgs]
    recon_b64 = [_flat_to_b64(r) for r in recs]

    # ── 2D PCA figure ──
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(
        x=p2[:, 0].tolist(), y=p2[:, 1].tolist(),
        mode='markers',
        marker=dict(size=5, color=color, opacity=0.65,
                    line=dict(width=0.3, color='white')),
        customdata=list(range(n_samples)),
        hovertemplate='PC1: %{x:.3f}<br>PC2: %{y:.3f}<extra></extra>'
    ))
    fig2.update_layout(
        title=dict(
            text=(f'2D PCA — Digit {digit}<br>'
                  f'<sub>Variance: {pca2.explained_variance_ratio_.sum():.1%}</sub>'),
            x=0.5, font=dict(size=15)),
        xaxis_title=f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})',
        yaxis_title=f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})',
        width=380, height=430,
        template='plotly_white',
        margin=dict(l=60, r=10, t=80, b=50),
        showlegend=False
    )

    # ── 2D UMAP figure ──
    fig_u = go.Figure()
    fig_u.add_trace(go.Scatter(
        x=u2[:, 0].tolist(), y=u2[:, 1].tolist(),
        mode='markers',
        marker=dict(size=5, color=color, opacity=0.65,
                    line=dict(width=0.3, color='white')),
        customdata=list(range(n_samples)),
        hovertemplate='UMAP1: %{x:.3f}<br>UMAP2: %{y:.3f}<extra></extra>'
    ))
    fig_u.update_layout(
        title=dict(
            text=f'2D UMAP — Digit {digit}',
            x=0.5, font=dict(size=15)),
        xaxis_title='UMAP 1',
        yaxis_title='UMAP 2',
        width=380, height=430,
        template='plotly_white',
        margin=dict(l=60, r=10, t=80, b=50),
        showlegend=False
    )

    # ── HTML fragments ──
    h2 = fig2.to_html(include_plotlyjs='cdn', div_id='dspec_pca2d', full_html=False)
    hu = fig_u.to_html(include_plotlyjs=False, div_id='dspec_umap2d', full_html=False)

    # JS image arrays (indexed by sample number)
    js_o = "[" + ",".join(f'"data:image/png;base64,{b}"' for b in orig_b64) + "]"
    js_r = "[" + ",".join(f'"data:image/png;base64,{b}"' for b in recon_b64) + "]"

    html = f"""
    <div style="display:flex; gap:10px; align-items:flex-start; flex-wrap:nowrap;">
      <div style="flex-shrink:0;">{h2}</div>
      <div style="flex-shrink:0;">{hu}</div>
    </div>
    <div id="dspec_tooltip" style="position:fixed; display:none; pointer-events:none; z-index:9999;
         background:white; border-radius:10px; box-shadow:0 4px 20px rgba(0,0,0,0.3);
         padding:8px 12px; text-align:center;"></div>
    <script>
    (function() {{
      var O={js_o};
      var R={js_r};
      var C="{color}";
      var tip=null;
      function getTip() {{
        if(!tip) tip=document.getElementById('dspec_tooltip');
        return tip;
      }}
      function show(i, evt) {{
        var t=getTip(); if(!t||!evt) return;
        t.innerHTML='<div>'
          +'<div style="display:flex;gap:12px;justify-content:center;">'
          +'<div><p style="margin:0 0 4px 0;font-weight:bold;color:#555;font-size:11px;">Original</p>'
          +'<img src="'+O[i]+'" style="border:2px solid '+C+';border-radius:6px;width:84px;height:84px;"></div>'
          +'<div><p style="margin:0 0 4px 0;font-weight:bold;color:#555;font-size:11px;">Reconstruction</p>'
          +'<img src="'+R[i]+'" style="border:2px solid '+C+';border-radius:6px;width:84px;height:84px;"></div>'
          +'</div></div>';
        t.style.display='block';
        var w=t.offsetWidth, h=t.offsetHeight;
        var cx=evt.clientX, cy=evt.clientY;
        var x=cx-w/2, y=cy-h-18;
        if(x<5) x=5;
        if(x+w>window.innerWidth-5) x=window.innerWidth-w-5;
        if(y<5) y=cy+18;
        t.style.left=x+'px'; t.style.top=y+'px';
      }}
      function hide() {{
        var t=getTip(); if(t) t.style.display='none';
      }}
      function attach() {{
        var a=document.getElementById('dspec_pca2d');
        var b=document.getElementById('dspec_umap2d');
        if(a&&a.on&&b&&b.on) {{
          a.on('plotly_hover',function(d){{var i=d.points[0].customdata;if(i!=null)show(i,d.event);}});
          b.on('plotly_hover',function(d){{var i=d.points[0].customdata;if(i!=null)show(i,d.event);}});
          a.on('plotly_unhover',function(){{hide();}});
          b.on('plotly_unhover',function(){{hide();}});
        }} else {{ setTimeout(attach,500); }}
      }}
      attach();
    }})();
    </script>
    """
    return html, pca2, n_samples


# ── Widget setup ──
_dspec_dd = widgets.Dropdown(
    options=[(f'Digit {d}', d) for d in range(10)],
    value=0,
    description='Select digit:',
    style={'description_width': 'initial'},
    layout={'width': '200px'}
)
_dspec_out = widgets.Output()

def _on_dspec_change(change):
    with _dspec_out:
        clear_output(wait=True)
        d = change['new']
        print(f"Generating visualization for digit {d}...")
        h, p2, n = _build_digit_viz(d)
        print(f"  PCA 2D variance explained: {p2.explained_variance_ratio_.sum():.2%}")
        print(f"  Samples: {n}")
        display(HTML(h))

_dspec_dd.observe(_on_dspec_change, names='value')
display(_dspec_dd)
display(_dspec_out)

# Initial render
with _dspec_out:
    print("Generating visualization for digit 0...")
    h, p2, n = _build_digit_viz(0)
    print(f"  PCA 2D variance explained: {p2.explained_variance_ratio_.sum():.2%}")
    print(f"  Samples: {n}")
    display(HTML(h))

### Data Augmentation Visualization
Apply rotations (±15°) and translations (±2px) to a selected digit class and visualize the augmented samples in the same latent space (PCA 2D, UMAP 2D) alongside the originals. Hover over any point to see the original/augmented image and its reconstruction.

In [ ]:
# ==================================================================
# DATA AUGMENTATION VISUALIZATION
# Apply rotations & translations to a digit class and visualize
# original vs augmented samples in the same latent space.
# ==================================================================
import numpy as np
import torch
import umap
from sklearn.decomposition import PCA
import plotly.graph_objects as go
from IPython.display import HTML, display, clear_output
import base64, io
from PIL import Image
import ipywidgets as widgets
from torchvision import transforms

# ── Augmentation transforms ──────────────────────────────────────────────────
_aug_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomAffine(
        degrees=15,              # rotations up to ±15°
        translate=(2/28, 2/28)   # translations up to ±2px
    ),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])


def _augment_batch(images_flat, n_augmented_per_image=3):
    """Given (N, 784) numpy float32 images, produce augmented copies.
    Returns (M, 784) numpy array of augmented images."""
    augmented = []
    for img in images_flat:
        img_2d = torch.FloatTensor(img).reshape(1, 28, 28)
        for _ in range(n_augmented_per_image):
            aug = _aug_transforms(img_2d)
            augmented.append(aug.numpy())
    return np.array(augmented, dtype=np.float32)


def _flat_to_b64_aug(flat_img):
    """Convert a flat (784,) float image to a base64 PNG string."""
    arr = (flat_img.reshape(28, 28) * 255).astype(np.uint8)
    pil = Image.fromarray(arr).resize((112, 112), Image.LANCZOS)
    buf = io.BytesIO()
    pil.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode()


def _build_aug_viz(digit):
    """Build PCA 2D + UMAP 2D for original & augmented samples of one digit."""
    ae = digit_autoencoders[digit]
    ae.eval()

    # ── Collect original test samples ────────────────────────────────────────
    digit_idx = (test_set.targets == digit).nonzero(as_tuple=True)[0].tolist()
    n_orig = min(500, len(digit_idx))
    chosen = np.random.choice(digit_idx, n_orig, replace=False)
    orig_imgs = np.array([test_set[idx][0].numpy() for idx in chosen])

    # ── Generate augmented images ────────────────────────────────────────────
    print(f"  Generating augmented samples (3x per original)...")
    aug_imgs = _augment_batch(orig_imgs, n_augmented_per_image=3)
    n_aug = len(aug_imgs)
    print(f"  Original: {n_orig}, Augmented: {n_aug}")

    # ── Combine and encode ───────────────────────────────────────────────────
    all_imgs = np.concatenate([orig_imgs, aug_imgs], axis=0)
    is_augmented = np.array([False]*n_orig + [True]*n_aug)

    t = torch.FloatTensor(all_imgs).to(device)
    with torch.no_grad():
        enc = ae.encoder(t)
        dec = ae.decoder(enc)
    acts = enc.cpu().numpy()
    recs = dec.cpu().numpy()

    # ── PCA 2D ───────────────────────────────────────────────────────────────
    pca2 = PCA(n_components=2).fit(acts)
    p2 = pca2.transform(acts)

    # ── UMAP 2D ──────────────────────────────────────────────────────────────
    print(f"  Computing UMAP 2D embedding...")
    reducer = umap.UMAP(n_components=2, random_state=42)
    u2 = reducer.fit_transform(acts)

    # ── Build base64 images ──────────────────────────────────────────────────
    n_total = len(all_imgs)
    print(f"  Converting {n_total} images to base64...")
    input_b64 = [_flat_to_b64_aug(im) for im in all_imgs]
    recon_b64 = [_flat_to_b64_aug(r) for r in recs]

    # ── Masks ────────────────────────────────────────────────────────────────
    mask_orig = ~is_augmented
    mask_aug = is_augmented

    color_orig = '#1f77b4'   # blue for originals
    color_aug = '#ff7f0e'    # orange for augmented

    # ── PCA figure ───────────────────────────────────────────────────────────
    fig_pca = go.Figure()
    fig_pca.add_trace(go.Scatter(
        x=p2[mask_orig, 0].tolist(), y=p2[mask_orig, 1].tolist(),
        mode='markers',
        marker=dict(size=5, color=color_orig, opacity=0.6,
                    line=dict(width=0.3, color='white')),
        customdata=np.where(mask_orig)[0].tolist(),
        name='Original',
        hovertemplate='PC1: %{x:.3f}<br>PC2: %{y:.3f}<extra>Original</extra>'
    ))
    fig_pca.add_trace(go.Scatter(
        x=p2[mask_aug, 0].tolist(), y=p2[mask_aug, 1].tolist(),
        mode='markers',
        marker=dict(size=4, color=color_aug, opacity=0.45,
                    symbol='diamond',
                    line=dict(width=0.3, color='white')),
        customdata=np.where(mask_aug)[0].tolist(),
        name='Augmented',
        hovertemplate='PC1: %{x:.3f}<br>PC2: %{y:.3f}<extra>Augmented</extra>'
    ))
    fig_pca.update_layout(
        title=dict(
            text=(f'2D PCA — Digit {digit} (Original vs Augmented)<br>'
                  f'<sub>Variance: {pca2.explained_variance_ratio_.sum():.1%}</sub>'),
            x=0.5, font=dict(size=14)),
        xaxis_title=f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})',
        yaxis_title=f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})',
        width=440, height=460,
        template='plotly_white',
        margin=dict(l=60, r=10, t=90, b=50),
        legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)')
    )

    # ── UMAP figure ──────────────────────────────────────────────────────────
    fig_umap = go.Figure()
    fig_umap.add_trace(go.Scatter(
        x=u2[mask_orig, 0].tolist(), y=u2[mask_orig, 1].tolist(),
        mode='markers',
        marker=dict(size=5, color=color_orig, opacity=0.6,
                    line=dict(width=0.3, color='white')),
        customdata=np.where(mask_orig)[0].tolist(),
        name='Original',
        hovertemplate='UMAP1: %{x:.3f}<br>UMAP2: %{y:.3f}<extra>Original</extra>'
    ))
    fig_umap.add_trace(go.Scatter(
        x=u2[mask_aug, 0].tolist(), y=u2[mask_aug, 1].tolist(),
        mode='markers',
        marker=dict(size=4, color=color_aug, opacity=0.45,
                    symbol='diamond',
                    line=dict(width=0.3, color='white')),
        customdata=np.where(mask_aug)[0].tolist(),
        name='Augmented',
        hovertemplate='UMAP1: %{x:.3f}<br>UMAP2: %{y:.3f}<extra>Augmented</extra>'
    ))
    fig_umap.update_layout(
        title=dict(
            text=f'2D UMAP — Digit {digit} (Original vs Augmented)',
            x=0.5, font=dict(size=14)),
        xaxis_title='UMAP 1',
        yaxis_title='UMAP 2',
        width=440, height=460,
        template='plotly_white',
        margin=dict(l=60, r=10, t=90, b=50),
        legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)')
    )

    # ── Compose HTML (same pattern as digit-specific viz above) ──────────────
    h_pca = fig_pca.to_html(include_plotlyjs='cdn', div_id='aug_pca2d', full_html=False)
    h_umap = fig_umap.to_html(include_plotlyjs=False, div_id='aug_umap2d', full_html=False)

    js_o = "[" + ",".join(f'"data:image/png;base64,{b}"' for b in input_b64) + "]"
    js_r = "[" + ",".join(f'"data:image/png;base64,{b}"' for b in recon_b64) + "]"
    js_aug = "[" + ",".join(str(int(a)) for a in is_augmented) + "]"

    html = f"""
    <div style="display:flex; gap:10px; align-items:flex-start; flex-wrap:nowrap;">
      <div style="flex-shrink:0;">{h_pca}</div>
      <div style="flex-shrink:0;">{h_umap}</div>
    </div>
    <div id="aug_tooltip" style="position:fixed; display:none; pointer-events:none; z-index:9999;
         background:white; border-radius:10px; box-shadow:0 4px 20px rgba(0,0,0,0.3);
         padding:8px 12px; text-align:center;"></div>
    <script>
    (function() {{
      var O={js_o};
      var R={js_r};
      var A={js_aug};
      var tip=null;
      function getTip() {{
        if(!tip) tip=document.getElementById('aug_tooltip');
        return tip;
      }}
      function show(i, evt) {{
        var t=getTip(); if(!t||!evt) return;
        var label=A[i]?'Augmented':'Original';
        var border=A[i]?'#ff7f0e':'#1f77b4';
        t.innerHTML='<div>'
          +'<p style="margin:0 0 6px;font-weight:bold;color:'+border+';font-size:12px;">'+label+'</p>'
          +'<div style="display:flex;gap:12px;justify-content:center;">'
          +'<div><p style="margin:0 0 4px 0;font-weight:bold;color:#555;font-size:11px;">Input</p>'
          +'<img src="'+O[i]+'" style="border:2px solid '+border+';border-radius:6px;width:84px;height:84px;"></div>'
          +'<div><p style="margin:0 0 4px 0;font-weight:bold;color:#555;font-size:11px;">Reconstruction</p>'
          +'<img src="'+R[i]+'" style="border:2px solid '+border+';border-radius:6px;width:84px;height:84px;"></div>'
          +'</div></div>';
        t.style.display='block';
        var w=t.offsetWidth, h=t.offsetHeight;
        var cx=evt.clientX, cy=evt.clientY;
        var x=cx-w/2, y=cy-h-18;
        if(x<5) x=5;
        if(x+w>window.innerWidth-5) x=window.innerWidth-w-5;
        if(y<5) y=cy+18;
        t.style.left=x+'px'; t.style.top=y+'px';
      }}
      function hide() {{
        var t=getTip(); if(t) t.style.display='none';
      }}
      function attach() {{
        var a=document.getElementById('aug_pca2d');
        var b=document.getElementById('aug_umap2d');
        if(a&&a.on&&b&&b.on) {{
          a.on('plotly_hover',function(d){{var i=d.points[0].customdata;if(i!=null)show(i,d.event);}});
          b.on('plotly_hover',function(d){{var i=d.points[0].customdata;if(i!=null)show(i,d.event);}});
          a.on('plotly_unhover',function(){{hide();}});
          b.on('plotly_unhover',function(){{hide();}});
        }} else {{ setTimeout(attach,500); }}
      }}
      attach();
    }})();
    </script>
    """
    return html, pca2, n_orig, n_aug


# ── Widget setup ──────────────────────────────────────────────────────────────
_aug_dd = widgets.Dropdown(
    options=[(f'Digit {d}', d) for d in range(10)],
    value=0,
    description='Select digit:',
    style={'description_width': 'initial'},
    layout={'width': '200px'}
)
_aug_out = widgets.Output()

def _on_aug_change(change):
    with _aug_out:
        clear_output(wait=True)
        d = change['new']
        print(f"Generating augmentation visualization for digit {d}...")
        h, p2, n_o, n_a = _build_aug_viz(d)
        print(f"  PCA 2D variance explained: {p2.explained_variance_ratio_.sum():.2%}")
        print(f"  Original samples: {n_o}, Augmented samples: {n_a}")
        display(HTML(h))

_aug_dd.observe(_on_aug_change, names='value')
display(_aug_dd)
display(_aug_out)

# Initial render
with _aug_out:
    print("Generating augmentation visualization for digit 0...")
    h, p2, n_o, n_a = _build_aug_viz(0)
    print(f"  PCA 2D variance explained: {p2.explained_variance_ratio_.sum():.2%}")
    print(f"  Original samples: {n_o}, Augmented samples: {n_a}")
    display(HTML(h))

### Digit-Specific Autoencoders Trained on Augmented Data
Train a new set of digit-specific autoencoders where the training data includes the original images **plus** augmented copies (rotations ±15°, translations ±2px). This lets us compare how augmentation during training affects the learned latent space and reconstruction quality.

In [ ]:
# ==================================================================
# TRAIN DIGIT-SPECIFIC AUTOENCODERS ON AUGMENTED DATA
# For each digit: original training images + 3 augmented copies each
# ==================================================================
from torchvision import transforms as T

_train_aug = T.Compose([
    T.ToPILImage(),
    T.RandomAffine(degrees=15, translate=(2/28, 2/28)),
    T.ToTensor(),
    T.Lambda(lambda x: x.view(-1))
])

digit_autoencoders_aug = {}
EPOCHS_DIGIT_AUG = 1
DIGIT_LAYERS_AUG = [28*28, 256, 128, 64, 10]

for digit in range(10):
    print(f"\n{'='*55}")
    print(f"  Training AUGMENTED autoencoder for digit {digit}")
    print(f"{'='*55}")

    # Collect original training images for this digit
    digit_indices = (training_set.targets == digit).nonzero(as_tuple=True)[0].tolist()
    orig_imgs = torch.stack([training_set[i][0] for i in digit_indices])  # (N, 784)

    # Generate augmented copies (3 per original)
    print(f"  Augmenting {len(digit_indices)} images (3x)...")
    aug_list = []
    for img in orig_imgs:
        img_2d = img.reshape(1, 28, 28)
        for _ in range(3):
            aug_list.append(_train_aug(img_2d))
    aug_imgs = torch.stack(aug_list)  # (3*N, 784)

    # Combine original + augmented
    all_imgs = torch.cat([orig_imgs, aug_imgs], dim=0)
    print(f"  Total training samples: {len(all_imgs)} "
          f"({len(orig_imgs)} orig + {len(aug_imgs)} aug)")

    # Build a DataLoader from the combined dataset
    combined_dataset = torch.utils.data.TensorDataset(all_imgs, all_imgs)
    aug_loader = torch.utils.data.DataLoader(
        combined_dataset, batch_size=64, shuffle=True, num_workers=2
    )

    # Create and train
    model_aug = Autoencoder(DIGIT_LAYERS_AUG, nn.ReLU).to(device)
    loss_fn_aug = nn.MSELoss()
    optimizer_aug = optim.Adam(model_aug.parameters(), lr=1e-3, weight_decay=1e-8)

    total_batches = EPOCHS_DIGIT_AUG * len(aug_loader)
    with tqdm(total=total_batches, desc=f"Digit {digit} (aug)") as pbar:
        for epoch in range(EPOCHS_DIGIT_AUG):
            model_aug.train()
            running_loss = 0.
            for i, (inputs, _) in enumerate(aug_loader):
                inputs = inputs.to(device)
                optimizer_aug.zero_grad()
                outputs = model_aug(inputs)
                loss = loss_fn_aug(outputs, inputs)
                loss.backward()
                optimizer_aug.step()
                running_loss += loss.item()
                pbar.update(1)
                pbar.set_postfix(loss=f"{running_loss / (i + 1):.5f}")
            model_aug.eval()

    digit_autoencoders_aug[digit] = model_aug
    print(f"  Final loss: {running_loss / (i + 1):.5f}")

print(f"\n{'='*55}")
print("  All 10 augmented digit-specific autoencoders trained!")
print(f"{'='*55}")

In [ ]:
# ==================================================================
# INTERACTIVE VISUALIZATION — Augmented-Trained Autoencoders
# Same PCA 2D + UMAP 2D + hover viz, now using digit_autoencoders_aug
# Also shows original vs augmented samples in the latent space.
# ==================================================================
import numpy as np
import torch
import umap
from sklearn.decomposition import PCA
import plotly.graph_objects as go
from IPython.display import HTML, display, clear_output
import base64, io
from PIL import Image
import ipywidgets as widgets
from torchvision import transforms

_aug_viz_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomAffine(degrees=15, translate=(2/28, 2/28)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])


def _flat_to_b64_augviz(flat_img):
    """Convert a flat (784,) float image to a base64 PNG string."""
    arr = (flat_img.reshape(28, 28) * 255).astype(np.uint8)
    pil = Image.fromarray(arr).resize((112, 112), Image.LANCZOS)
    buf = io.BytesIO()
    pil.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode()


def _build_aug_trained_viz(digit):
    """Build PCA 2D + UMAP 2D for the augmented-trained autoencoder."""
    ae = digit_autoencoders_aug[digit]
    ae.eval()

    # ── Collect original test samples ────────────────────────────────────────
    digit_idx = (test_set.targets == digit).nonzero(as_tuple=True)[0].tolist()
    n_orig = min(500, len(digit_idx))
    chosen = np.random.choice(digit_idx, n_orig, replace=False)
    orig_imgs = np.array([test_set[idx][0].numpy() for idx in chosen])

    # ── Generate augmented test images ───────────────────────────────────────
    print(f"  Generating augmented test samples (3x per original)...")
    aug_list = []
    for img in orig_imgs:
        img_2d = torch.FloatTensor(img).reshape(1, 28, 28)
        for _ in range(3):
            aug_list.append(_aug_viz_transforms(img_2d).numpy())
    aug_imgs = np.array(aug_list, dtype=np.float32)
    n_aug = len(aug_imgs)
    print(f"  Original: {n_orig}, Augmented: {n_aug}")

    # ── Combine and encode ───────────────────────────────────────────────────
    all_imgs = np.concatenate([orig_imgs, aug_imgs], axis=0)
    is_augmented = np.array([False]*n_orig + [True]*n_aug)

    t = torch.FloatTensor(all_imgs).to(device)
    with torch.no_grad():
        enc = ae.encoder(t)
        dec = ae.decoder(enc)
    acts = enc.cpu().numpy()
    recs = dec.cpu().numpy()

    # ── PCA 2D ───────────────────────────────────────────────────────────────
    pca2 = PCA(n_components=2).fit(acts)
    p2 = pca2.transform(acts)

    # ── UMAP 2D ──────────────────────────────────────────────────────────────
    print(f"  Computing UMAP 2D embedding...")
    reducer = umap.UMAP(n_components=2, random_state=42)
    u2 = reducer.fit_transform(acts)

    # ── Build base64 images ──────────────────────────────────────────────────
    n_total = len(all_imgs)
    print(f"  Converting {n_total} images to base64...")
    input_b64 = [_flat_to_b64_augviz(im) for im in all_imgs]
    recon_b64 = [_flat_to_b64_augviz(r) for r in recs]

    # ── Masks ────────────────────────────────────────────────────────────────
    mask_orig = ~is_augmented
    mask_aug = is_augmented

    color_orig = '#2ca02c'   # green for originals
    color_aug = '#d62728'    # red for augmented

    # ── PCA figure ───────────────────────────────────────────────────────────
    fig_pca = go.Figure()
    fig_pca.add_trace(go.Scatter(
        x=p2[mask_orig, 0].tolist(), y=p2[mask_orig, 1].tolist(),
        mode='markers',
        marker=dict(size=5, color=color_orig, opacity=0.6,
                    line=dict(width=0.3, color='white')),
        customdata=np.where(mask_orig)[0].tolist(),
        name='Original',
        hovertemplate='PC1: %{x:.3f}<br>PC2: %{y:.3f}<extra>Original</extra>'
    ))
    fig_pca.add_trace(go.Scatter(
        x=p2[mask_aug, 0].tolist(), y=p2[mask_aug, 1].tolist(),
        mode='markers',
        marker=dict(size=4, color=color_aug, opacity=0.45,
                    symbol='diamond',
                    line=dict(width=0.3, color='white')),
        customdata=np.where(mask_aug)[0].tolist(),
        name='Augmented',
        hovertemplate='PC1: %{x:.3f}<br>PC2: %{y:.3f}<extra>Augmented</extra>'
    ))
    fig_pca.update_layout(
        title=dict(
            text=(f'2D PCA — Digit {digit} (Aug-Trained)<br>'
                  f'<sub>Variance: {pca2.explained_variance_ratio_.sum():.1%}</sub>'),
            x=0.5, font=dict(size=14)),
        xaxis_title=f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})',
        yaxis_title=f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})',
        width=440, height=460,
        template='plotly_white',
        margin=dict(l=60, r=10, t=90, b=50),
        legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)')
    )

    # ── UMAP figure ──────────────────────────────────────────────────────────
    fig_umap = go.Figure()
    fig_umap.add_trace(go.Scatter(
        x=u2[mask_orig, 0].tolist(), y=u2[mask_orig, 1].tolist(),
        mode='markers',
        marker=dict(size=5, color=color_orig, opacity=0.6,
                    line=dict(width=0.3, color='white')),
        customdata=np.where(mask_orig)[0].tolist(),
        name='Original',
        hovertemplate='UMAP1: %{x:.3f}<br>UMAP2: %{y:.3f}<extra>Original</extra>'
    ))
    fig_umap.add_trace(go.Scatter(
        x=u2[mask_aug, 0].tolist(), y=u2[mask_aug, 1].tolist(),
        mode='markers',
        marker=dict(size=4, color=color_aug, opacity=0.45,
                    symbol='diamond',
                    line=dict(width=0.3, color='white')),
        customdata=np.where(mask_aug)[0].tolist(),
        name='Augmented',
        hovertemplate='UMAP1: %{x:.3f}<br>UMAP2: %{y:.3f}<extra>Augmented</extra>'
    ))
    fig_umap.update_layout(
        title=dict(
            text=f'2D UMAP — Digit {digit} (Aug-Trained)',
            x=0.5, font=dict(size=14)),
        xaxis_title='UMAP 1',
        yaxis_title='UMAP 2',
        width=440, height=460,
        template='plotly_white',
        margin=dict(l=60, r=10, t=90, b=50),
        legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)')
    )

    # ── Compose HTML ─────────────────────────────────────────────────────────
    h_pca = fig_pca.to_html(include_plotlyjs='cdn', div_id='augtrain_pca2d', full_html=False)
    h_umap = fig_umap.to_html(include_plotlyjs=False, div_id='augtrain_umap2d', full_html=False)

    js_o = "[" + ",".join(f'"data:image/png;base64,{b}"' for b in input_b64) + "]"
    js_r = "[" + ",".join(f'"data:image/png;base64,{b}"' for b in recon_b64) + "]"
    js_aug = "[" + ",".join(str(int(a)) for a in is_augmented) + "]"

    html = f"""
    <div style="display:flex; gap:10px; align-items:flex-start; flex-wrap:nowrap;">
      <div style="flex-shrink:0;">{h_pca}</div>
      <div style="flex-shrink:0;">{h_umap}</div>
    </div>
    <div id="augtrain_tooltip" style="position:fixed; display:none; pointer-events:none; z-index:9999;
         background:white; border-radius:10px; box-shadow:0 4px 20px rgba(0,0,0,0.3);
         padding:8px 12px; text-align:center;"></div>
    <script>
    (function() {{
      var O={js_o};
      var R={js_r};
      var A={js_aug};
      var tip=null;
      function getTip() {{
        if(!tip) tip=document.getElementById('augtrain_tooltip');
        return tip;
      }}
      function show(i, evt) {{
        var t=getTip(); if(!t||!evt) return;
        var label=A[i]?'Augmented':'Original';
        var border=A[i]?'#d62728':'#2ca02c';
        t.innerHTML='<div>'
          +'<p style="margin:0 0 6px;font-weight:bold;color:'+border+';font-size:12px;">'+label+'</p>'
          +'<div style="display:flex;gap:12px;justify-content:center;">'
          +'<div><p style="margin:0 0 4px 0;font-weight:bold;color:#555;font-size:11px;">Input</p>'
          +'<img src="'+O[i]+'" style="border:2px solid '+border+';border-radius:6px;width:84px;height:84px;"></div>'
          +'<div><p style="margin:0 0 4px 0;font-weight:bold;color:#555;font-size:11px;">Reconstruction</p>'
          +'<img src="'+R[i]+'" style="border:2px solid '+border+';border-radius:6px;width:84px;height:84px;"></div>'
          +'</div></div>';
        t.style.display='block';
        var w=t.offsetWidth, h=t.offsetHeight;
        var cx=evt.clientX, cy=evt.clientY;
        var x=cx-w/2, y=cy-h-18;
        if(x<5) x=5;
        if(x+w>window.innerWidth-5) x=window.innerWidth-w-5;
        if(y<5) y=cy+18;
        t.style.left=x+'px'; t.style.top=y+'px';
      }}
      function hide() {{
        var t=getTip(); if(t) t.style.display='none';
      }}
      function attach() {{
        var a=document.getElementById('augtrain_pca2d');
        var b=document.getElementById('augtrain_umap2d');
        if(a&&a.on&&b&&b.on) {{
          a.on('plotly_hover',function(d){{var i=d.points[0].customdata;if(i!=null)show(i,d.event);}});
          b.on('plotly_hover',function(d){{var i=d.points[0].customdata;if(i!=null)show(i,d.event);}});
          a.on('plotly_unhover',function(){{hide();}});
          b.on('plotly_unhover',function(){{hide();}});
        }} else {{ setTimeout(attach,500); }}
      }}
      attach();
    }})();
    </script>
    """
    return html, pca2, n_orig, n_aug


# ── Widget setup ──────────────────────────────────────────────────────────────
_augtrain_dd = widgets.Dropdown(
    options=[(f'Digit {d}', d) for d in range(10)],
    value=0,
    description='Select digit:',
    style={'description_width': 'initial'},
    layout={'width': '200px'}
)
_augtrain_out = widgets.Output()

def _on_augtrain_change(change):
    with _augtrain_out:
        clear_output(wait=True)
        d = change['new']
        print(f"Generating visualization for aug-trained digit {d}...")
        h, p2, n_o, n_a = _build_aug_trained_viz(d)
        print(f"  PCA 2D variance explained: {p2.explained_variance_ratio_.sum():.2%}")
        print(f"  Original samples: {n_o}, Augmented samples: {n_a}")
        display(HTML(h))

_augtrain_dd.observe(_on_augtrain_change, names='value')
display(_augtrain_dd)
display(_augtrain_out)

# Initial render
with _augtrain_out:
    print("Generating visualization for aug-trained digit 0...")
    h, p2, n_o, n_a = _build_aug_trained_viz(0)
    print(f"  PCA 2D variance explained: {p2.explained_variance_ratio_.sum():.2%}")
    print(f"  Original samples: {n_o}, Augmented samples: {n_a}")
    display(HTML(h))